# S3_NB4 — does the memorisation collapse damage pruning?

**~18 GPU-hours · 14 runs · fully resumable · independent of Q1 and Q2**

## The claim being tested

Study 2 §4.3: softmax difficulty scores lose seed reliability on data the model
has **memorised** — `ce_loss` ρ_seed falls 0.647 → 0.108 — predicted by
saturation (ρ = +0.832) and **not** by test accuracy (−0.114).

Dataset pruning computes exactly these scores, on exactly this data, from a
single seed. **A warning is not a demonstration.** This measures the damage.

| source | train acc | > 0.99 conf | ρ_seed drop |
|---|---|---|---|
| `convnext_femto` (saturated) | 99.99 % | 71 % | 0.558 |
| `mobilenetv2` (unsaturated) | 83.1 % | 16 % | 0.026 |

**Pre-registered (H3):** the saturated source is worse by **≥ 1.0 pt** at 30 %
retention, and the gap widens as retention falls.
**H3b:** the saturated source is indistinguishable from **random** pruning
(±0.5 pt) at 30 %. If true, that is the headline.

## The cheap gate runs first

If the two sources keep nearly the same samples, no downstream difference is
possible and 18 GPU-hours buy nothing. That check costs minutes.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   a1ecac2d9d61   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  2cc4ba5e0935   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
    'bGluZSBjaGVja3MiKQo=',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

In [ ]:
# === CELL 2 -- WHERE EVERYTHING LIVES ======================================
# Leave both as None and they are CHOSEN FOR YOU: the roomiest drive that
# actually exists on this machine. Set MSC_ROOT explicitly if your runs are
# somewhere specific -- e.g. r'C:\msc_results'.

DATA_DIR = None      # e.g. r'E:\msc_data'      -- None = choose for me
MSC_ROOT = None      # e.g. r'C:\msc_results'   -- None = choose for me

# WHERE CIFAR-100 ALREADY IS. Point this at the folder that contains
# `cifar-100-python` (or at that folder itself -- both work). It is checked
# BEFORE any download path, so nothing ever re-fetches 169 MB over a copy you
# already have.
CIFAR_DIR = r'C:\Users\Administrator\Desktop\New folder'

# ---------------------------------------------------------------------------
import os
from pathlib import Path

M = msc                       # Study 2's notebooks say `M`; same module.

if CIFAR_DIR:
    os.environ['MSC_CIFAR_DIR'] = str(CIFAR_DIR)

# Study 3 is CIFAR-100. The library's default repo is the ImageNet one, so
# without this every push would land in msc-imagenet100 -- which is exactly what
# happened on the first joint run.
os.environ.setdefault('MSC_HF_REPO', 'Shanmuk4622/msc-cifar100')

_paths = M.resolve_storage(DATA_DIR, MSC_ROOT)
if not _paths['ok']:
    raise SystemExit('storage is not usable -- see the problems listed above')

DATA_DIR = _paths['data_dir']
MSC_ROOT = _paths['results_root']
os.environ['MSC_SCRATCH'] = MSC_ROOT

# OFFLINE BY DEFAULT. This machine does not always have a network, and a
# background uploader that retries mid-epoch turns a missing connection into a
# failed run. Everything is written to disk in full; S3_NB5_Publish uploads it
# in ONE pass at the end. `enable_hf` is the only switch that decides this.
sess = M.Session(account='local', phase='p5', dataset='cifar100',
                 work_root=MSC_ROOT, session_limit_h=0.0,
                 enable_hf=False,
                 worker_id=0, num_workers=1)
print('HuggingFace: ' + ('ON -- publishing' if False else
                         'OFF -- fully offline, nothing is uploaded here'))

print(f'msc_lib   {M.__version__}')
print(f'MSC_ROOT  {MSC_ROOT}')
print(f'data_dir  {sess.data_dir}')

# Resolve CIFAR-100 now, loudly, rather than discovering mid-training that it
# is about to download. `_has_cifar100` is the same check the loader uses.
if CIFAR_DIR:
    _cd = Path(CIFAR_DIR)
    _cd = _cd if M._has_cifar100(_cd) else _cd.parent
    if M._has_cifar100(_cd):
        print(f'CIFAR-100  {_cd}  (found -- no download)')
    else:
        print(f'CIFAR-100  NOT at {CIFAR_DIR} -- expected a cifar-100-python/'
              ' folder with train/ and test/ inside. It will try to download.')

# A Session CREATES runs/, so "the directory exists" proves nothing. Count the
# runs that actually carry a measurement -- that is what every cell below reads.
_runs_dir = Path(MSC_ROOT) / 'runs'
_measured = sorted(d.name for d in _runs_dir.iterdir()
                   if d.is_dir() and (d / 'per_sample' / 'test.parquet').exists()
                   ) if _runs_dir.is_dir() else []
print(f'measured runs on disk: {len(_measured)}')
if not _measured:
    raise RuntimeError(
        f'no measured runs under {MSC_ROOT}/runs.\n'
        'Study 3 re-analyses Study 1 output. Fetch it first with\n'
        'notebooks_study2/S2_NB0_Fetch.ipynb, or set MSC_ROOT above\n'
        'to the folder that already holds them.')
print(f'  e.g. {_measured[0]}')

In [ ]:
# === Which runs am I analysing? ===========================================
# One accessor, so the dedupe rule and the emptiness checks live in a single
# place rather than being re-typed in each notebook (rule 4).
import numpy as np, pandas as pd

runs_dir = Path(MSC_ROOT) / 'runs'

def measured_runs(dataset='cifar100', methods=('base',), require=True):
    ids = sorted(d.name for d in runs_dir.iterdir()
                 if d.is_dir() and (d / 'per_sample' / 'test.parquet').exists())
    if not ids:
        raise RuntimeError(f'no measured runs under {runs_dir}')
    df = pd.DataFrame([{**M.parse_run_id(r), 'run_id': r} for r in ids])

    for col in ('method', 'dataset', 'arch', 'seed', 'phase'):
        if col not in df.columns:
            raise RuntimeError(
                f'parse_run_id did not yield a {col!r} column. Columns present: '
                f'{list(df.columns)}. Run ids look like: {ids[:3]}')

    sel = df[(df['dataset'] == dataset) & (df['method'].isin(methods))]
    if require and sel.empty:
        raise RuntimeError('; '.join([
            f'{len(df)} measured run(s) on disk, but NONE with '
            f'dataset={dataset!r} and method in {tuple(methods)!r}',
            f'datasets present: {sorted(df["dataset"].dropna().unique())}',
            f'methods present: {sorted(df["method"].dropna().unique())}',
            'either the wrong MSC_ROOT is set, or the runs you need have not '
            'been fetched or trained yet']))

    # p0 pilots and p1 runs share seed numbers, so pooling them counts one seed
    # twice -- the contamination Study 2 found. Keep the highest phase.
    before = len(sel)
    sel = (sel.sort_values('phase')
              .drop_duplicates(subset=['arch', 'dataset', 'method', 'seed'],
                               keep='last'))
    if len(sel) < before:
        print(f'dropped {before - len(sel)} duplicate (arch, method, seed) '
              f'run(s) -- pilot replicates')
    return sel.reset_index(drop=True)

_b = measured_runs()
print(f'{len(_b)} CIFAR-100 base run(s); '
      f'{_b["arch"].nunique()} architecture(s)')

In [ ]:
# === Get CIFAR-100 =========================================================
# Looks in this order: attached Kaggle dataset (instant) -> earlier extraction
# -> Kaggle CLI download -> torchvision as a last resort.
# Everything lands in /kaggle/temp (~1 TB scratch), never in /kaggle/working
# (20 GB, and that is the space your results need).
DATA_ROOT = sess.prepare_data()
print('dataset:', DATA_ROOT)

---
## Gate — do the two sources even disagree?

In [ ]:

import numpy as np, pandas as pd
from pathlib import Path

SATURATED   = 'convnext_femto'
UNSATURATED = 'mobilenetv2'
RATES       = [0.5, 0.3]
TARGET_ARCH = 'resnet20'
SCORE       = 'ce_loss'

runs_dir = Path(MSC_ROOT) / 'runs'

def train_score(arch, seed=1):
    hits = [d.name for d in runs_dir.iterdir()
            if d.is_dir() and f'-{arch}-cifar100-base-s{seed}' in d.name
            and (d / 'per_sample' / 'train_holdout.parquet').exists()]
    if not hits:
        raise RuntimeError(f'no train_holdout parquet for {arch} seed {seed}')
    d = pd.read_parquet(runs_dir / sorted(hits)[-1] / 'per_sample'
                        / 'train_holdout.parquet')
    return d.set_index('sample_idx')[SCORE].astype(float)

s_sat = train_score(SATURATED)
s_uns = train_score(UNSATURATED)
common = s_sat.index.intersection(s_uns.index)
print(f'{len(common):,} samples scored by both sources')

print()
print('kept-set overlap (keeping the HARDEST samples):')
overlaps = {}
for rate in RATES:
    n_keep = int(len(common) * rate)
    a = set(s_sat.loc[common].nlargest(n_keep).index)
    b = set(s_uns.loc[common].nlargest(n_keep).index)
    ov = len(a & b) / n_keep
    overlaps[rate] = ov
    print(f'    keep {rate:.0%}:  {ov:.1%} of the kept samples are shared')

if min(overlaps.values()) >= 0.90:
    raise RuntimeError(
        f'overlap >= 90% at every rate ({overlaps}). The two sources disagree '
        'on RELIABILITY but agree on WHICH SAMPLES TO KEEP, so no downstream '
        'difference is possible. CANCEL this experiment and report the overlap '
        'itself -- that is a finding, and it cost minutes instead of 18 GPU-h.')
print()
print('the sources disagree enough for a downstream difference to be possible')

---
## Build the subsets

Three arms per retention rate: saturated-guided, unsaturated-guided, random.
Plus a full-data control. Two seeds each for the trained arms.

In [ ]:

import json
subset_dir = Path(MSC_ROOT) / 'subsets'
subset_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(0)
specs = []
for rate in RATES:
    n_keep = int(len(common) * rate)
    keeps = {
        'sat':  sorted(int(i) for i in s_sat.loc[common].nlargest(n_keep).index),
        'uns':  sorted(int(i) for i in s_uns.loc[common].nlargest(n_keep).index),
        'rand': sorted(int(i) for i in rng.choice(np.asarray(common),
                                                  n_keep, replace=False)),
    }
    for arm, idx in keeps.items():
        p = subset_dir / f'keep_{arm}_{int(rate*100)}.json'
        p.write_text(json.dumps({'arm': arm, 'rate': rate, 'score': SCORE,
                                 'n_keep': len(idx), 'keep': idx}))
        specs.append({'arm': arm, 'rate': rate, 'path': str(p), 'n': len(idx)})

print(pd.DataFrame(specs)[['arm', 'rate', 'n']].to_string(index=False))
print()
print(f'{len(specs)} subset(s) written to {subset_dir}')

---
## Train the target on each subset

`resnet20`, ~1.3 GPU-h per run. **Resumable and safe to stop** — same machinery
as every other training notebook in this project.

`subset_path` is part of the config, so it is inside `config_hash`: two arms can
never silently share a checkpoint.

In [ ]:

SEEDS = [1, 2]
cfgs = []
for sp in specs:
    for sd in SEEDS:
        cfgs.append(sess.config(
            TARGET_ARCH, seed=sd,
            method=f"prune{sp['arm']}{int(sp['rate']*100)}",
            subset_path=sp['path']))
# full-data controls
for sd in SEEDS:
    cfgs.append(sess.config(TARGET_ARCH, seed=sd, method='prunefull'))

for c in cfgs:
    print(f"  {c['run_id']:44s} subset={Path(c.get('subset_path', '-')).name}")
print(f'\n{len(cfgs)} run(s)')

# The library must actually honour subset_path. If it silently ignores an
# unknown config key, every arm trains on the full dataset and the experiment
# returns a perfect null that looks like a result.
import inspect
_src = inspect.getsource(M.build_loaders)
if 'subset_path' not in _src:
    raise RuntimeError(
        'build_loaders does not read subset_path. Every arm would train on the '
        'FULL dataset and produce identical results -- a null that looks like a '
        'finding. Implement subset support before spending 18 GPU-hours.')
print('build_loaders honours subset_path')

In [ ]:

ok = all(M.backbone_dry_run(c) for c in cfgs[:2])
print('dry run ok' if ok else 'DRY RUN FAILED')

results = sess.run_all(cfgs, title='Study 3 Q3 / pruning')
for r in results:
    print(f"  {r.get('status','?'):9s} {r.get('run_id','?')}  "
          f"top1={M.fmt_metric(r.get('best_accuracy'))}")

---
## H3 — the verdict

In [ ]:

rows = []
for c in cfgs:
    L = M.run_layout(sess.work, c['run_id'])
    s = L['base'] / 'summary.json'
    if not s.exists():
        continue
    d = json.loads(s.read_text())
    m = M.parse_run_id(c['run_id'])
    meth = m['method']
    arm = ('full' if meth == 'prunefull'
           else ''.join(ch for ch in meth[5:] if not ch.isdigit()))
    rate = ''.join(ch for ch in meth if ch.isdigit()) or '100'
    rows.append({'arm': arm, 'rate': int(rate), 'seed': m['seed'],
                 'acc': d.get('best_accuracy') or d.get('final_acc')})

pr = pd.DataFrame(rows).dropna()
M.save_analysis(sess.data_dir, 's3_pruning', pr)
tab = pr.groupby(['rate', 'arm'])['acc'].agg(['mean', 'std', 'count'])
print((tab * 100).round(2).to_string())

print()
for rate in sorted(pr['rate'].unique()):
    if rate == 100:
        continue
    sub = pr[pr['rate'] == rate]
    g = sub.groupby('arm')['acc'].mean() * 100
    if 'sat' in g and 'uns' in g:
        gap = g['uns'] - g['sat']
        print(f'  keep {rate}%:  unsaturated {g["uns"]:.2f}  '
              f'saturated {g["sat"]:.2f}  gap {gap:+.2f} pt', end='')
        if 'rand' in g:
            print(f'   random {g["rand"]:.2f}  '
                  f'(saturated - random {g["sat"]-g["rand"]:+.2f})')
        else:
            print()

print()
print('H3  (unsaturated beats saturated by >= 1.0 pt at 30% retention)')
print('H3b (saturated indistinguishable from random, +/- 0.5 pt at 30%)')
print()
print('If H3b holds, the headline is: a single-seed difficulty score from a')
print('memorising model is worth no more than chance.')